# Validation by Comparison with FEniCS

This notebook presents how the validation suite compares SOFA with [FEniCS](https://fenicsproject.org), and walks through the code that does it on one case.

## Validation in a nutshell

A case is a boundary-value problem solved twice: once by SOFA, with the component under test, and once by FEniCS, an independent finite element library. Both solve it on the same mesh, so both give a displacement at the same nodes, and the two are compared node by node. Where the problem has an exact solution, SOFA is compared with it as well.

### Measuring the difference

The difference is measured by its relative RMS over the $N$ nodes of the mesh, where $\mathbf{u}_i$ is SOFA's displacement at node $i$ and $\mathbf{u}_i^\text{ref}$ the reference one, from FEniCS or the exact solution:

$$e_\text{rms} = \sqrt{\frac{\sum_{i=1}^{N} \|\mathbf{u}_i - \mathbf{u}_i^\text{ref}\|^2}{\sum_{i=1}^{N} \|\mathbf{u}_i^\text{ref}\|^2}}$$

A case passes when $e_\text{rms}$ against FEniCS is below a set `tolerance`, and, if the case has an exact solution, $e_\text{rms}$ against it is below its `toleranceAnalytic`.

## Example: axial compression of a 2D beam

The rest of this notebook walks through one case, `validation/cases/LinearSmallStrainFEMForceField/2D/AxialCompression/`, building each piece of it in turn.

### The problem

The beam starts at the origin, with length $L$ along $x$ and width $W$ along $y$. Its right side carries a uniform compressive traction $p$. The left side is held only along $x$, so that the beam can contract freely across its width, and the stress is uniform, $\sigma_{xx} = -p$. The exact solution is

$$u_x = -\frac{p}{E}\,x, \qquad u_y = \frac{\nu p}{E}\,y$$

- `left` ($x = 0$): $u_x = 0$
- corner $(0, 0)$: $u_y = 0$
- `right` ($x = L$): $\mathbf{t} = (-p, 0)$
- `bottom` and `top`: $\mathbf{t} = 0$

### The case folder

A case is a folder, named after the problem and placed under its component and dimension:

```
AxialCompression/
├── case.json                 the input file, with one entry per element
├── sofa_scene.py             builds the problem in SOFA and solves it
├── fenics_scene.py           builds the same problem in FEniCS and solves it
├── reference_solution.json   FEniCS's solution, written by fenics_scene.py
└── analytic_solution.py      the exact solution, for the cases that have one
```

Each of these is introduced below, where the walkthrough reaches it.

### The input file

The problem is defined once, in `case.json`, and both `sofa_scene.py` and `fenics_scene.py` read it. The geometry, mesh, material and load come from this one place, so the information SOFA and FEniCS share cannot drift apart. This departs a bit from how SOFA scenes are usually written, with their values set in the scene itself.

`case.json` holds one entry per element the case is run on, here `quad` and `tri`. The walkthrough uses the `quad` entry, below. Its keys:

- `geometry` names the geometry and gives its size, $L$ and $W$;
- `element` is the kind of element the geometry is meshed with;
- `material` names the constitutive law and gives its parameters, $E$ and $\nu$;
- `forceField` is the SOFA component under test;
- `solvers` sets how SOFA solves the discrete system;
- `mesh` gives the mesh resolution;
- `load` gives the traction $p$;
- `tolerance` and `toleranceAnalytic` are the thresholds on $e_\text{rms}$.

In [ ]:
case = {
    "geometry":   {"type": "Beam2D", "length": 1.0, "width": 0.2},
    "element":    "quad",
    "material":   {"type": "LinearElastic", "youngModulus": 1000.0, "poissonRatio": 0.3},
    "forceField": {"type": "LinearSmallStrainFEMForceField"},
    "solvers": {
        "integration":  {"type": "StaticEquilibriumIntegrationScheme", "maxNbIterationsNewton": 1,
                         "residueThreshold": 0},
        "linearSolver": {"type": "SparseLDLSolver", "template": "CompressedRowSparseMatrixd"},
    },
    "mesh":       {"resolution": [21, 9, 1]},
    "load":       {"type": "axialTraction", "magnitude": 200.0},
    "tolerance":         {"rms": 1e-10},
    "toleranceAnalytic": {"rms": 1e-10},
}

### SOFA's solution

`sofa_scene.py` builds the problem as a SOFA scene and solves it. Its `solve(case)` returns the nodes $\mathbf{x}$ of the mesh at rest and SOFA's displacement $\mathbf{u}$ at each of them, then the polygons the figures are drawn on: the elements in 1D and 2D, the boundary faces in 3D.

The file is also a SOFA scene, to inspect a case in the SOFA GUI:

```bash
runSofa -l SofaPython3 validation/cases/LinearSmallStrainFEMForceField/2D/AxialCompression/sofa_scene.py --argv quad
```

In [ ]:
import pathlib
import sys

# Make the plugin's packages, then the case's modules, importable from validation/docs
sys.path.insert(0, "../..")
CASE_DIR = pathlib.Path("../cases/LinearSmallStrainFEMForceField/2D/AxialCompression")
sys.path.insert(0, str(CASE_DIR))

import sofa_scene

x, u, polygons = sofa_scene.solve(case)

### The FEniCS reference

`fenics_scene.py` solves the same problem with FEniCS. Since it reads the same `case.json` entry, the geometry, mesh resolution, material and load are SOFA's. It also builds the same mesh as SOFA, with the same nodes and elements, so the two solutions compare node by node.

`fenics_scene.py` writes FEniCS's nodes and displacements to `reference_solution.json`, one entry per element. The file is committed with the case, so FEniCS is needed only to regenerate it (`--fenics` below). The suite checks that the nodes match before it compares the displacements.

In [ ]:
import json

import numpy as np

reference = json.loads((CASE_DIR / "reference_solution.json").read_text())["quad"]["fenics"]
x_fenics, u_fenics = np.array(reference["x"]), np.array(reference["u"])

assert np.allclose(x, x_fenics, atol=1e-8)

### Comparing

$e_\text{rms}$ is computed by `RMSRelative` (`validation/study/metrics.py`). The exact solution is the `displacement(x, case)` function of the case's `analytic_solution.py`.

The exact displacement is linear in $x$ and $y$, and linear elements reproduce it exactly, so both differences are at round-off level. This is why this case's tolerances are $10^{-10}$.

In [ ]:
from analytic_solution import displacement
from validation.study.metrics import RMSRelative

u_exact = displacement(x, case)
rms = RMSRelative()
print(f"e_rms against FEniCS:             {rms.measure(u, u_fenics):.2e} (tolerance {case['tolerance']['rms']:.0e})")
print(f"e_rms against the exact solution: {rms.measure(u, u_exact):.2e} (tolerance {case['toleranceAnalytic']['rms']:.0e})")

## Putting it together

Everything the walkthrough did by hand is done by `ISFComparisonStudy` (`validation/study/study.py`), for every element of `case.json`:

- it loads FEniCS's solution from `reference_solution.json`, running `fenics_scene.py` first if the file has no entry for the element;
- it calls `solve(case)` of `sofa_scene.py`, and checks that SOFA's nodes are FEniCS's;
- it measures $e_\text{rms}$ against FEniCS and, if `analytic_solution.py` exists, against the exact solution, and decides whether the element passes;
- it draws a figure per element: a row each for SOFA, FEniCS and the exact solution, a column per component of $\mathbf{u}$.

`run` (`validation/run.py`) runs the study and saves its figures under `validation/results/`.

In [ ]:
from validation.run import run

study = run(CASE_DIR)
for element in study.results:
    print(f"{element}: e_rms = {study.results[element]['rms']:.2e} against FEniCS, "
          f"{study.analytic_results[element]['rms']:.2e} against the exact solution, "
          f"{'pass' if study.passed[element] else 'fail'}")

study.plots["quad"]

The same runs from the command line:

```bash
python validation/run.py validation/cases/LinearSmallStrainFEMForceField/2D/AxialCompression
python validation/run.py --all       # every case, then one summary line per case and element
python validation/run.py --all --fenics   # regenerate every reference_solution.json first
```

The cases of a component are presented in its own notebook, e.g. [LinearSmallStrainFEMForceField.ipynb](LinearSmallStrainFEMForceField.ipynb).